# Dev31: Quick Plotting Tests (NO PIPELINE REQUIRED!)

This notebook loads pre-saved classification results so you can test plotting functions **instantly** without running the entire classification pipeline.

**Workflow:**
1. Run the "Save Results" cells in dev30 (one time only)
2. Then use this notebook to test plotting changes instantly!

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import matplotlib.patches as mpatches
import pickle
import os

print("✅ Imports ready")

## Load Saved Classification Results

In [ ]:
# Load the saved classification results
save_file = "./saved_data/classification_results_028.pkl"

if not os.path.exists(save_file):
    print("❌ ERROR: Save file not found!")
    print(f"   Looking for: {save_file}")
    print("\n📝 TO CREATE THE SAVE FILE:")
    print("   1. Open dev30 notebook")
    print("   2. Run all cells up to classification")
    print("   3. Run the 'Save Results' cell at the bottom")
    print("   4. Then come back here!")
else:
    with open(save_file, "rb") as f:
        saved_data = pickle.load(f)

    multiclass_crop_results = saved_data["multiclass_crop_results"]
    crop_regions = saved_data["crop_regions"]

    print("✅ Classification results loaded!")
    print(f"   {len(multiclass_crop_results)} crop results")
    print(f"\n📍 Crop regions:")
    for i, crop in enumerate(crop_regions, 1):
        print(
            f"   {i}. {crop['name']}: track={crop['track']}, slit={crop['slit']}, width={crop['width']}"
        )

## Define Plotting Function

In [ ]:
def plot_classification_as_image(
    crop,
    classification_map,
    class_names,
    figsize=(30, 8.32),
    flip_axes=True,
    flip_horizontal=True,
):
    """
    Plot classification results as a proper classified image.
    Each pixel gets the color of its class - like a normal image.
    """
    # Define colors for each class
    class_colors = {
        "sediment": [0.6, 0.4, 0.2],  # Brown
        "rust": [0.8, 0.2, 0.1],  # Red-orange
        "dark_bomb": [0.1, 0.1, 0.1],  # Dark gray/black
        "dark_pit": [0.2, 0.2, 0.2],  # Gray
        "halo": [0.9, 0.9, 0.5],  # Yellow
        "uncertain": [0.5, 0.5, 0.5],  # Medium gray
        "classified_unknown": [0.5, 0.5, 0.5],  # Same as uncertain
    }

    # Create RGB image from classification map
    height, width = classification_map.shape
    rgb_image = np.zeros((height, width, 3))

    for class_name in class_names:
        mask = classification_map == class_name
        color = class_colors.get(class_name, [0.5, 0.5, 0.5])
        rgb_image[mask] = color

    # Apply flipping to match plot_rgb behavior
    # Step 1: Transpose to match plot_rgb's initial .T.copy()
    rgb_image = rgb_image.transpose(1, 0, 2)  # (slits, tracks, 3)

    # Step 2: Apply flip_axes (like plot_rgb does)
    if flip_axes:
        rgb_image = rgb_image.transpose(1, 0, 2)  # Back to (tracks, slits, 3)

    # Step 3: Apply flip_horizontal (like plot_rgb does)
    if flip_horizontal:
        rgb_image = np.flip(rgb_image, axis=1)  # Flip along axis 1

    # Calculate extent in global coordinates
    half_width_slit = crop["width"] // 2
    half_width_track = int(crop["width"] / crop["aspect_ratio"] / 2)

    slit_min = crop["slit"] - half_width_slit
    slit_max = crop["slit"] + half_width_slit
    track_min = crop["track"] - half_width_track
    track_max = crop["track"] + half_width_track

    if flip_axes:
        # When axes flipped: x=slit, y=track (matches plot_rgb)
        extent = [slit_min, slit_max, track_min, track_max]
        xlabel, ylabel = "Slit Pixel Index", "Track Index"
    else:
        # Normal: x=track, y=slit
        extent = [track_min, track_max, slit_min, slit_max]
        xlabel, ylabel = "Track Index", "Slit Pixel Index"

    # Create figure
    fig, ax = plt.subplots(figsize=figsize)

    # Display the classified image
    im = ax.imshow(
        rgb_image,
        extent=extent,
        origin="lower",
        aspect=crop.get("display_aspect_ratio", 4.0),
        interpolation="nearest",
    )

    ax.set_xlabel(xlabel, fontsize=12)
    ax.set_ylabel(ylabel, fontsize=12)
    ax.set_title(f"Classification: {crop['name']}", fontsize=14, fontweight="bold")

    # Create legend with class colors
    legend_patches = []
    for class_name in class_names:
        count = np.sum(classification_map == class_name)
        color = class_colors.get(class_name, [0.5, 0.5, 0.5])
        label = f"{class_name} ({count} px)"
        legend_patches.append(mpatches.Patch(color=color, label=label))

    ax.legend(
        handles=legend_patches,
        loc="center left",
        bbox_to_anchor=(1, 0.5),
        frameon=True,
        fontsize=11,
    )

    plt.tight_layout()
    plt.show()

    return fig, ax


print("✅ plot_classification_as_image defined")

## 🎨 Plot All Classification Results

In [ ]:
%matplotlib inline

print("📊 Plotting all classification results...")

for crop_result in multiclass_crop_results:
    crop = crop_result['crop']
    results = crop_result['results']
    
    print(f"\n📍 {crop['name']}")
    print(f"   Classes: {results['class_names']}")
    
    # CRITICAL FIX: Crop the classification_map to the slit range!
    # classify_segment only crops track dimension, not slit dimension
    half_width_slit = crop['width'] // 2
    crop_slit_min = max(0, crop['slit'] - half_width_slit)
    crop_slit_max = min(results['classification_map'].shape[1], crop['slit'] + half_width_slit)
    
    # Crop the classification map to the slit range
    cropped_classification_map = results['classification_map'][:, crop_slit_min:crop_slit_max]
    
    print(f"   Original shape: {results['classification_map'].shape}")
    print(f"   Cropped to slit [{crop_slit_min}:{crop_slit_max}]: {cropped_classification_map.shape}")
    print(f"   Pixel counts:")
    for class_name in results['class_names']:
        count = np.sum(cropped_classification_map == class_name)
        percentage = 100.0 * count / cropped_classification_map.size
        print(f"      {class_name}: {count} pixels ({percentage:.1f}%)")
    
    # Plot as proper classified image with CROPPED data
    plot_classification_as_image(
        crop,
        cropped_classification_map,  # Use CROPPED classification map!
        results['class_names'],
        figsize=(30, 7.18),
        flip_axes=True,
        flip_horizontal=True
    )